In [ ]:
import re
import underthesea
import emot
import unicodedata
from pathlib import Path

In [ ]:
def lowercase(text):
    text = text.lower()
    return text

def remove_urls(text):
    tmp =  re.sub(r'https?://\S+', '', text)
    res = re.sub(r'www\.\S+', '', tmp)
    return res

def remove_punctuation(text):
    res = re.sub(r'[^\w\s]*', '', text)
    return res

def remove_digits(text):
    return re.sub(r'\d+', '', text)


def sentence_tokenizer(text):
    return underthesea.sent_tokenize(text)

emot_obj = emot.core.emot()
def convert_emotion(text):
    emoji = emot_obj.emoji(text)
    emoticons = emot_obj.emoticons(text)
    
    for i,j in zip(emoji['value'], emoji['mean']):
        text = text.replace(i,j.replace(':', ' '))
        
    for i,j in zip(emoticons['value'], emoticons['mean']):
        text = text.replace(i,'_'.join(tmp for tmp in j.split()))
    
    return text

def remove_stop_words(text):
    with open('vietnamese-stopwords.txt', 'r', encoding='utf-8') as f:
        stop_words = set(f.read().split('\n'))
    
    l = [i for i in text.split() if i not in stop_words]
    
    return ' '.join(l)

def standardize_text(text):
    return unicodedata.normalize("NFC", text)

def standardize_shortwords(text):
    d = {
        'k': 'không',
        'ko': 'không'
    }

    return ' '.join(d.get(i,i) for i in text.split())


def correct_spelling(text):
    return underthesea.text_normalize(text)

def word_tokenizer(text):
    return underthesea.word_tokenize(text)


In [ ]:
data_dir = Path("data")
out_dir = Path("Data_Preprocessing")
out_dir.mkdir(exist_ok=True)

# count=0
for path in data_dir.glob('*.txt'):
    text = path.read_text(encoding='utf-8')
    
    text = standardize_text(text)
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_digits(text)
    sentence_list = sentence_tokenizer(text)
    # print(sentence_list)

    # 1 file chỉ có 1 câu nên tách ra vẫn vậy
    new_text = sentence_list[0]
    new_text = convert_emotion(new_text)
    new_text = standardize_shortwords(new_text)
    new_text = remove_stop_words(new_text)
    new_text = correct_spelling(new_text)
    word_tokenize_list = word_tokenizer(new_text)
    
    word_tokenize_list = [t.replace(' ', '_') for t in word_tokenize_list]

    print(word_tokenize_list)

    out_path = out_dir / f"{path.stem}_preprocessed.txt"
    out_path.write_text(' '.join(word_tokenize_list), encoding="utf-8")